# SNC — Evaluate the Conditioned Separator

Runs every test the codebase ships with, against the latest trained
model on Drive. Use this after each training run to check that the
model has not collapsed and that detection / removal still work.

| Stage | Script | What it measures |
|------|--------|------------------|
| 1 | `diagnose_model.py` | Smoke test — is the model alive and class-conditional? |
| 2 | `evaluate_conditioned_separator.py` | SI-SDR / SI-SDRi on synthetic positive-only mixtures |
| 3 | `evaluate_detection.py` | Precision / Recall / F1 on synthetic multi-class mixtures |
| 4 | `test_realworld.py` | Detection + optional removal on a user-uploaded audio file |

Pick the **T4** GPU runtime (newer GPUs can fail with `CUDA_ERROR_INVALID_PTX`).

## 1. Mount Drive and clone the repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DRIVE_ROOT = '/content/drive/MyDrive/snc'
GITHUB_USER = 'keremtutumlu'
GITHUB_REPO_NAME = 'selective-noise-cancelling'
BRANCH = 'feature/separator-quality-overhaul'

# Private repo? Add a Colab Secret named GITHUB_TOKEN (key icon, scope: repo).
import os, subprocess, shutil
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

clone_url = (f'https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
             if token else
             f'https://github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git')

REPO_DIR = Path(f'/content/{GITHUB_REPO_NAME}')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
result = subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('git clone failed — if the repo is private add a '
                       'GITHUB_TOKEN secret in the left sidebar.')
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

Working dir: /content/selective-noise-cancelling


In [3]:
# === Model version — set once, applies to every evaluation stage below ===
# diagnose / SI-SDR / detection / real-world all read the SNC_MODEL_VERSION
# environment variable, so bumping this (e.g. 'v2.5') retargets the whole
# notebook with no code edits and no commits.
import os
os.environ['SNC_MODEL_VERSION'] = 'v2.5'
# Decoded-dataset cache lives on local SSD, not the Drive-backed data
# root: the first decode of ~10k WAVs from Drive can take over an hour,
# but it is pickled once here and every later script reads it in seconds.
os.environ['SNC_DATA_CACHE_DIR'] = '/content'
print('Model version for this run:', os.environ['SNC_MODEL_VERSION'])
print('Dataset cache dir       :', os.environ['SNC_DATA_CACHE_DIR'])

Model version for this run: v2.5
Dataset cache dir       : /content


In [4]:
# Symlink data and saved_models to Drive so the trained model and datasets
# are accessible without re-downloading.
drive_data = Path(DRIVE_ROOT) / 'data'
drive_models = Path(DRIVE_ROOT) / 'saved_models'

for local, target in [(REPO_DIR / 'data', drive_data),
                       (REPO_DIR / 'saved_models', drive_models)]:
    if local.is_symlink() or local.exists():
        local.unlink() if local.is_symlink() else shutil.rmtree(local)
    local.symlink_to(target)
    print(f'{local} -> {target}')

version = os.environ.get('SNC_MODEL_VERSION', 'v2.4')
ckpt = drive_models / 'separation_models' / f'separator_unet_film_multi_{version}.h5'
assert ckpt.exists(), f'Trained model not found at {ckpt} — run training first.'
print('Model found:', ckpt)

/content/selective-noise-cancelling/data -> /content/drive/MyDrive/snc/data
/content/selective-noise-cancelling/saved_models -> /content/drive/MyDrive/snc/saved_models
Model found: /content/drive/MyDrive/snc/saved_models/separation_models/separator_unet_film_multi_v2.5.h5


In [5]:
!pip install -q librosa==0.11.0 soundfile scikit-learn pandas
import tensorflow as tf
print('TF version :', tf.__version__)
print('GPUs       :', tf.config.list_physical_devices('GPU'))

TF version : 2.20.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Smoke test — is the model alive?

Checks three things in under a minute:

1. Model produces non-zero output for known present classes.
2. Different class queries on the same input produce **different** outputs (FiLM is doing something).
3. Across many classes, the correct query consistently beats wrong queries.

If this fails, the model has collapsed and there is no point running the
longer evaluations — retrain with safer hyperparameters first.

In [6]:
!python src/model_training/diagnose_model.py


Model diagnostic — separator_unet_film_multi_v2.5.h5
Data root    — /content/selective-noise-cancelling/data/raw

2026-06-05 11:13:20.196595: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780658000.198012   10087 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-06-05 11:35:34,826 - INFO - ESC-50: 2000 clips, 50 classes.
2026-06-05 13:18:06,868 - INFO - UrbanSound8K: 8732 clips, 10 classes.
2026-06-05 13:18:06,870 - INFO - Merged: 56 classes, 10732 clips total.
2026-06-05 13:18:12,426 - INFO - Cached decoded dataset to _decoded_cache_v1_sr16000_min40.pkl (delete to rebuild).
2026-06-05 13:18:12,427 - INFO - SeparationMixer ready: 56 classes.
Model vocabulary: 227 classes; 56 have local audio for t

## 3. SI-SDR — separation quality on synthetic mixtures

Builds 800 positive-only test mixtures (the query class is always
present), runs separation, and scores SI-SDR against the ground-truth
stem. The headline metric is **SI-SDRi** — improvement over the
unprocessed mixture.

Healthy ranges:
- **+5 dB or more** — working
- **0–2 dB** — under-trained
- **≤ 0 dB** — model is making it worse (collapsed)

In [7]:
!python src/model_training/evaluate_conditioned_separator.py


Conditioned Separator — Evaluation Report
Model : /content/selective-noise-cancelling/saved_models/separation_models/separator_unet_film_multi_v2.5.h5

2026-06-05 13:18:32,232 - INFO - Loaded decoded dataset from cache: _decoded_cache_v1_sr16000_min40.pkl (56 classes, 10732 clips).
2026-06-05 13:18:32,232 - INFO - SeparationMixer ready: 56 classes.
2026-06-05 13:18:32.783016: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780665512.784530   42406 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model vocabulary: 227 classes; 56 have local audio for mixtures.

Running separation on 800 test mixtures... 2026-06-05 13:18:45.540068: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kern

## 4. Detection — precision / recall on synthetic multi-class mixtures

Builds 200 mixtures of 1–4 random classes, runs the webapp detection
scoring, and reports per-class precision, recall, F1. This tests the
detect_sounds path independently from separation.

Look for mean F1 of at least **0.5** — below that the surfaced classes
are largely noise.

In [8]:
!python src/model_training/evaluate_detection.py --write-allowlist

2026-06-05 13:19:00,107 - INFO - Loaded decoded dataset from cache: _decoded_cache_v1_sr16000_min40.pkl (56 classes, 10732 clips).
2026-06-05 13:19:00,108 - INFO - SeparationMixer ready: 56 classes.
Wrote 56-class detection allow-list to /content/selective-noise-cancelling/saved_models/separation_models/separator_unet_film_multi_v2.5_detect_allowlist.json

Detection Evaluation — separator_unet_film_multi_v2.5.h5
Test mixtures: 200
Thresholds   : floor=0.05  relative_cap=0.8  top_k=5

2026-06-05 13:19:01.000019: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780665541.001892   42639 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-06-05 13:19:04,310 - INFO - Loaded decoded dataset from cache: _decod

In [9]:
!cd /content/selective-noise-cancelling && git pull --rebase

!python src/model_training/evaluate_detection.py --relative-cap 0.65 --top-k 10

!python src/model_training/evaluate_detection.py --relative-cap 0.80 --top-k 5

!python src/model_training/evaluate_detection.py --relative-cap 0.90 --top-k 3

error: cannot pull with rebase: You have unstaged changes.
error: please commit or stash them.

Detection Evaluation — separator_unet_film_multi_v2.5.h5
Test mixtures: 200
Thresholds   : floor=0.05  relative_cap=0.65  top_k=10

2026-06-05 13:22:31.598203: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780665751.599718   47629 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-06-05 13:22:36,192 - INFO - Loaded decoded dataset from cache: _decoded_cache_v1_sr16000_min40.pkl (56 classes, 10732 clips).
2026-06-05 13:22:36,193 - INFO - SeparationMixer ready: 56 classes.
Model vocabulary: 227 classes; 56 have local audio for mixtures.
Detection allow-list: 56 classes (only these can be surfaced).

2026-06